# PTM Corpus Build + ESM-2 Activation Extraction Pipeline
### Corpus/Dataset Creation, Activation Sharding, and Hugging Face + Kaggle Dataset Sync

This notebook runs the full data pipeline on Kaggle (or locally), in two phases:

**Phase A -- Corpus/Dataset Creation** (`ptm_sae.corpus.pipeline`, ported from the partner's N1 methodology):
1. **Acquire & Filter**: Downloads the Swiss-Prot human proteome + 6 PTM databases (CPLM, dbPTM, qPTM, O-GlcNAcAtlas, N-GlycositeAtlas, N-GlyDE).
2. **Cluster & Split**: CD-HIT clustering at 40% identity, subdivided into the discovery_train/discovery_val/held_out partition.
3. **Label Cascade**: Isoform recovery, residue-chemistry-validated PTM canonicalization, ambiguity masking, cross-source dedup.
4. **Publish**: Uploads corpus.parquet/labels_stratified.parquet (+ companions) to Hugging Face and publishes a Kaggle Dataset.

**Phase B -- ESM-2 Activation Extraction** (`run_full_lifecycle`):
1. **Read**: Loads the already-published corpus's discovery_train/discovery_val proteins.
2. **Extract & Shard**: Extracts ESM-2 residual activations into indexed SafeTensors shards.
3. **Sync & Verify**: Uploads shards to Hugging Face Hub and verifies zero-copy tensor deserialization.

### Structured Hugging Face Repository Layout:
```
{user}/ptm-sae-dataset/
├── corpus/                                 <- Phase A output (Parquet, not JSONL)
│   ├── corpus.parquet                      <- Sequences + cluster_id + partition (discovery_train/discovery_val/held_out)
│   ├── labels_stratified.parquet           <- Validated multi-label PTM sites
│   ├── stratum_counts.json, split_manifest.json
│   ├── exclusion_mask.parquet
│   └── gold_negatives_nglyco.parquet
│
└── activations/
    └── esm2_t33_650M_UR50D/
        └── layer_24/
            ├── manifest.json               <- Activation shard index & offsets
            ├── mean_pooled_embeddings.safetensors  <- Auxiliary sequence context
            └── shards/                     <- Clean directory containing only shards
                ├── shard_0000.safetensors
                ├── shard_0001.safetensors
                └── ...
```

In [ ]:
# 1. Environment & GPU Verification
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(
        f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )
!nvidia-smi

In [ ]:
# 2. Repository Setup (same on a local PC, a persistent remote box, Kaggle and Colab)
# Finds the checkout by walking up from the kernel's working directory, so it also works when this
# notebook is opened from notebooks/. Only a runtime with no checkout (fresh Kaggle/Colab) clones;
# an existing checkout is fast-forwarded (local edits are auto-stashed, never discarded) on its
# current branch -- switch branches yourself with `git checkout` in a terminal.
import os
import subprocess
import sys
from pathlib import Path

BRANCH = os.environ.get("PTM_SAE_BRANCH", "main")  # used only when cloning a fresh runtime
PULL = True  # fast-forward an existing checkout on every run
REPO = "github.com/Mumu-kun/PTM-SAE.git"

here = Path.cwd().resolve()
root = next(
    (p for p in (here, *here.parents) if (p / "pyproject.toml").is_file() and (p / "src/ptm_sae").is_dir()),
    None,
)

if root is None:
    token = os.environ.get("GH_TOKEN")  # private repo: also read Kaggle/Colab secrets
    try:
        from kaggle_secrets import UserSecretsClient

        token = token or UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata

        token = token or userdata.get("GH_TOKEN")
    except Exception:
        pass

    auth = f"x-access-token:{token}@" if token else ""
    root = (Path("/kaggle/working") if Path("/kaggle").exists() else here) / "PTM-SAE"
    print(f"Cloning {BRANCH} into {root}...")
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://{auth}{REPO}", str(root)], check=True)
elif PULL:
    print(f"Fast-forwarding existing checkout at {root}...")
    subprocess.run(["git", "pull", "--ff-only", "--autostash"], cwd=root, check=False)

os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from ptm_sae import runtime

print(f"Platform: {runtime.detect_platform()} | repo: {root} | data root: {runtime.resolve_data_root()}")

In [ ]:
# 3. Install Dependencies (single-sourced from pyproject.toml)
# Local/remote boxes manage their environment with `uv sync --group notebook` in a terminal (locked
# versions, CUDA torch build kept intact), so this cell only checks it. Kaggle/Colab ship a
# CUDA-matched torch/transformers, so there everything else is pip-installed and those two are left alone.
import subprocess
import sys

EXTRAS = ("kaggle",)

if runtime.detect_platform() == "local":
    missing = runtime.missing_requirements(extras=EXTRAS)
    assert not missing, f"Missing {missing}: run `uv sync --group notebook` in a terminal, then restart the kernel."
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *runtime.cloud_requirements(extras=EXTRAS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
# 3b. cd-hit (system package, not pip -- only needed for Phase A)
import shutil

if shutil.which("cd-hit") is None:
    !apt-get install -y -q cd-hit

In [ ]:
# 4. Authentication Discovery (Zero-Knowledge Secrets)

from ptm_sae.extraction.hub import resolve_hf_token

token = resolve_hf_token()
if token:
    print(f"[Auth] HF_TOKEN discovered successfully (prefix: {token[:6]}...).")
else:
    print(
        "[Auth] Notice: No HF_TOKEN detected. Remote uploads will be disabled unless set in Kaggle/Colab Secrets."
    )

In [ ]:
# 4b. Kaggle API Credential Discovery (needed for Phase A's Kaggle Dataset publish)
import os

if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
    print(
        f"[Auth] KAGGLE_USERNAME/KAGGLE_KEY discovered "
        f"(user: {os.environ['KAGGLE_USERNAME']})."
    )
else:
    print(
        "[Auth] Notice: No KAGGLE_USERNAME/KAGGLE_KEY detected. Set both in Kaggle Secrets "
        "(Add-ons -> Secrets) if you want Phase A to publish a Kaggle Dataset."
    )

In [ ]:
# 5. Phase A Params -- Corpus/Dataset Creation
# ---------------------------------------------------------------------------
MOCK = False  # True -> small synthetic fixtures, no network/cd-hit-scale data (local verification only)
FORCE = False  # True -> ignore every cache, rebuild from scratch
# ---------------------------------------------------------------------------

HF_CORPUS_REPO_ID = "mustafa-muhaimin/ptm-sae-corpus"  # corpus tables; activations go to REMOTE_REPO_ID (Phase B params)
HF_CORPUS_SUBPATH = "corpus"
KAGGLE_DATASET_SLUG = "<your-kaggle-username>/ptm-sae-corpus"  # fill in before running

print(f"Mock: {MOCK} | Force rebuild: {FORCE}")
print(f"HF target: {HF_CORPUS_REPO_ID}/{HF_CORPUS_SUBPATH}")
print(f"Kaggle Dataset target: {KAGGLE_DATASET_SLUG}")

In [ ]:
# 6. Run Corpus/Dataset Creation Pipeline
# (Acquire -> Cluster & Split -> Label Cascade -> Publish to HF + Kaggle Dataset)
# Calls the pipeline directly in this process -- no subprocess/!python, so CD-HIT's own
# progress and any exception traceback show up immediately, not buffered behind a
# subprocess pipe.

from ptm_sae.corpus import pipeline as corpus_pipeline
from ptm_sae.corpus.config import CorpusPaths
from ptm_sae.extraction.hub import publish_kaggle_dataset

corpus_result = corpus_pipeline.run(mock=MOCK, force=FORCE)
print(f"Corpus build complete: {corpus_result['corpus']['partition'].value_counts().to_dict()}")

corpus_paths = CorpusPaths.from_env()
print(f"\nUploading to HF ({HF_CORPUS_REPO_ID}/{HF_CORPUS_SUBPATH}):")
corpus_pipeline.upload_to_hf(corpus_paths, HF_CORPUS_REPO_ID, HF_CORPUS_SUBPATH)

if KAGGLE_DATASET_SLUG and "<your-kaggle-username>" not in KAGGLE_DATASET_SLUG:
    print(f"\nPublishing Kaggle Dataset ({KAGGLE_DATASET_SLUG}):")
    publish_kaggle_dataset(
        corpus_paths.processed_dir, KAGGLE_DATASET_SLUG, "PTM-SAE Corpus", "Corpus rebuild"
    )
else:
    print("\nKAGGLE_DATASET_SLUG not set, skipping Kaggle Dataset publish.")

In [ ]:
# 7. Phase B Params -- Activation Extraction
# ---------------------------------------------------------------------------
# Choose your execution profile:
#   PROFILE = "SAMPLE"     -> Uses data/sample.fasta & ESM-2 8M (2-minute sanity check)
#   PROFILE = "PILOT"      -> Streams UniProt, extracts first 500 proteins (~10 mins)
#   PROFILE = "PRODUCTION" -> Streams UniProt, extracts full ~16,000 Discovery proteins on 650M
# ---------------------------------------------------------------------------
PROFILE = "PILOT"  # Toggle between 'SAMPLE', 'PILOT', 'PRODUCTION'

# Your private Hugging Face Dataset repository (set to None for local-only testing)
REMOTE_REPO_ID = "mustafa-muhaimin/ptm-sae-dataset"  # e.g. "fahad/ptm-activations"

if PROFILE == "SAMPLE":
    CONFIG_PATH = "configs/dev_8m.yaml"
    SAMPLE_ONLY = True
    MAX_PROTEINS = None
elif PROFILE == "PILOT":
    CONFIG_PATH = "configs/dev_8m.yaml"
    SAMPLE_ONLY = False
    MAX_PROTEINS = 500  # Extract first 500 UniProt proteins
elif PROFILE == "PRODUCTION":
    CONFIG_PATH = "configs/colab_650m.yaml"
    SAMPLE_ONLY = False
    MAX_PROTEINS = (
        None  # Full Discovery partition (8,894 train + 3,240 val = 12,134 proteins)
    )

from ptm_sae.config import PipelineConfig

cfg = PipelineConfig.from_yaml(CONFIG_PATH)

cfg.sharding.output_dir = runtime.anchor_path(
    cfg.sharding.output_dir, runtime.resolve_data_root()
)

print(f"Profile: {PROFILE}")
print(
    f"Model: {cfg.model.model_name} | Target Layer: {cfg.model.target_layer} | Dtype: {cfg.model.dtype}"
)
print(f"Output directory: {cfg.sharding.output_dir}")
print(f"Remote target repo: {REMOTE_REPO_ID or 'Local storage only'}")
print(f"Remote subpath: {cfg.resolve_remote_subpath()}")

In [ ]:
# 8. Run ESM-2 Activation Extraction
# (reads the already-published corpus's discovery_train/discovery_val proteins -- Phase A must have run first -- then extracts, uploads shards, and verifies)
import os

from ptm_sae import run_full_lifecycle

results = run_full_lifecycle(
    config=cfg,
    sample_only=SAMPLE_ONLY,
    max_proteins=MAX_PROTEINS,
    remote_repo_id=REMOTE_REPO_ID,
    remote_corpus_repo_id=HF_CORPUS_REPO_ID,
    kaggle_username=os.environ.get("KAGGLE_USERNAME"),  # None -> Kaggle Dataset publish skipped
)

In [ ]:
# 9. Zero-Copy SafeTensors Readback Sanity Check

from ptm_sae.extraction import SafeTensorsReader

reader = SafeTensorsReader(
    cache_dir=cfg.sharding.output_dir,
    remote_repo_id=REMOTE_REPO_ID,
    remote_subpath=cfg.resolve_remote_subpath(),
)
proteins = reader.list_proteins()
print(f"Total accessible proteins: {len(proteins)}")
print(f"First 5 proteins: {proteins[:5]}")

if proteins:
    test_id = proteins[0]
    acts = reader.get_protein_activations(test_id)
    res1 = reader.get_residue_activation(test_id, 1)
    mean_vec = reader.get_mean_pooled_embedding(test_id)
    print(f"[Verification] {test_id} sequence tensor shape: {acts.shape}")
    print(f"[Verification] {test_id} residue 1 vector shape: {res1.shape}")
    print(
        f"[Verification] {test_id} mean-pooled context vector shape: {mean_vec.shape}"
    )
    print(
        "[SUCCESS] End-to-end biological coordinate invariants and SafeTensors zero-copy access verified!"
    )